# Assignment 3 — Module 2: R Programming
## Retail Data Integration, Revenue Analysis, Customer Segmentation & SQLite

**Dataset:** UCI Online Retail (`Online Retail.xlsx`)  
**Source:** The uploaded `online+retail.zip` contains the original UCI Excel dataset.

### Important
This notebook uses **only the uploaded UCI Online Retail dataset**. No synthetic transaction, product, or customer datasets are created.

Because Google Colab normally runs Python cells, this notebook executes the complete **R implementation through `Rscript`**. This avoids the `%%R` / `rpy2` errors that can occur in Colab.

### Assignment workflow
1. Upload `online+retail.zip`
2. Install R and required R packages
3. Extract and read the UCI Excel file
4. Clean the data
5. Derive transaction, product and customer tables from the same UCI source
6. Join the derived tables
7. Calculate revenue and perform business analysis
8. Segment customers using `case_when()`
9. Create a SQLite database and run SQL queries
10. Generate three business insights
11. Export all outputs and create one final submission ZIP


In [4]:
# CELL 1 — Install R and required R packages
# This cell is Python because Colab's default kernel is Python.
# The actual analysis is performed in R using Rscript.

import os, subprocess, textwrap, sys

print("Checking R installation...")

if subprocess.run(["bash", "-lc", "command -v Rscript"], capture_output=True, text=True).returncode != 0:
    print("Installing R...")
    subprocess.run(["bash", "-lc", "apt-get update -qq && apt-get install -y -qq r-base r-base-dev"], check=True)
else:
    print("R is already installed.")

r_install = r'''
options(repos = c(CRAN = "https://cloud.r-project.org"))

pkgs <- c(
  "readxl", "dplyr", "tidyr", "ggplot2", "readr",
  "jsonlite", "DBI", "RSQLite", "lubridate", "skimr"
)

installed <- rownames(installed.packages())
needed <- setdiff(pkgs, installed)

if (length(needed) > 0) {
  install.packages(needed, dependencies = TRUE, quiet = TRUE)
}

cat("R packages ready:\n")
print(pkgs)
'''
subprocess.run(["Rscript", "-e", r_install], check=True)
print("✓ R + packages are ready.")


Checking R installation...
R is already installed.
✓ R + packages are ready.


In [3]:
# CELL 2 — Upload the UCI Online Retail ZIP
from google.colab import files
import os, zipfile, glob

uploaded = files.upload()

zip_candidates = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_candidates:
    raise ValueError("Please upload the UCI file: online+retail.zip")

zip_name = zip_candidates[0]
print("Uploaded:", zip_name)

extract_dir = "/content/uci_online_retail"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(extract_dir)

print("\nExtracted files:")
for root, dirs, fs in os.walk(extract_dir):
    for f in fs:
        print(os.path.join(root, f))


Saving online+retail.zip to online+retail.zip
Uploaded: online+retail.zip

Extracted files:
/content/uci_online_retail/Online Retail.xlsx


In [5]:
# CELL 3 — Locate the original UCI Excel file
import os, glob

excel_files = []
for root, dirs, fs in os.walk("/content/uci_online_retail"):
    for f in fs:
        if f.lower().endswith((".xlsx", ".xls")):
            excel_files.append(os.path.join(root, f))

if not excel_files:
    raise FileNotFoundError("No Excel file was found inside the uploaded ZIP.")

excel_path = excel_files[0]
print("Using UCI dataset:")
print(excel_path)


Using UCI dataset:
/content/uci_online_retail/Online Retail.xlsx


In [6]:
# CELL 4 — Write the complete R implementation and execute it
# This is the main R implementation, executed safely through Rscript.

r_code = r'''
# ============================================================
# ASSIGNMENT 3 — MODULE 2
# UCI ONLINE RETAIL: CLEANING, JOINS, ANALYSIS, SQLITE
# ============================================================

suppressPackageStartupMessages({
  library(readxl)
  library(dplyr)
  library(tidyr)
  library(ggplot2)
  library(readr)
  library(DBI)
  library(RSQLite)
  library(lubridate)
  library(skimr)
})

# -----------------------------
# 0. Paths
# -----------------------------
excel_path <- Sys.getenv("UCI_EXCEL_PATH")
output_dir <- "/content/Assignment_3_Final_Deliverables"
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

cat("\n========================================\n")
cat("ASSIGNMENT 3 — UCI ONLINE RETAIL\n")
cat("========================================\n")
cat("Input:", excel_path, "\n")

# -----------------------------
# 1. Import
# -----------------------------
cat("\n[1] IMPORTING DATA\n")

retail_raw <- read_excel(excel_path)

cat("Rows:", nrow(retail_raw), "\n")
cat("Columns:", ncol(retail_raw), "\n")
cat("\nColumn names:\n")
print(names(retail_raw))

cat("\nFirst 6 rows:\n")
print(head(retail_raw))

# Standardize column names explicitly to the UCI names.
retail <- retail_raw %>%
  rename(
    InvoiceNo = InvoiceNo,
    StockCode = StockCode,
    Description = Description,
    Quantity = Quantity,
    InvoiceDate = InvoiceDate,
    UnitPrice = UnitPrice,
    CustomerID = CustomerID,
    Country = Country
  )

# -----------------------------
# 2. Initial data-quality summary
# -----------------------------
cat("\n[2] INITIAL DATA QUALITY\n")

missing_before <- data.frame(
  Variable = names(retail),
  Missing_Count = sapply(retail, function(x) sum(is.na(x))),
  Missing_Percentage = round(sapply(retail, function(x) mean(is.na(x)) * 100), 4)
)

duplicate_count_before <- sum(duplicated(retail))
cancellation_count <- sum(grepl("^C", as.character(retail$InvoiceNo)))

print(missing_before)
cat("\nDuplicate rows before cleaning:", duplicate_count_before, "\n")
cat("Cancellation rows:", cancellation_count, "\n")

# -----------------------------
# 3. Clean the actual UCI data
# -----------------------------
cat("\n[3] CLEANING DATA\n")

retail_clean <- retail %>%
  mutate(
    InvoiceNo = as.character(InvoiceNo),
    StockCode = as.character(StockCode),
    Description = trimws(as.character(Description)),
    Country = trimws(as.character(Country)),
    Quantity = as.numeric(Quantity),
    UnitPrice = as.numeric(UnitPrice),
    CustomerID = as.character(CustomerID),
    InvoiceDate = as.POSIXct(InvoiceDate)
  ) %>%
  # Remove duplicate records
  distinct() %>%
  # Remove rows without essential product/transaction information
  filter(!is.na(InvoiceNo),
         !is.na(StockCode),
         !is.na(Description),
         Description != "") %>%
  # Remove cancellations. UCI marks cancellations with InvoiceNo beginning C.
  filter(!grepl("^C", InvoiceNo)) %>%
  # Remove invalid quantities and prices
  filter(!is.na(Quantity), Quantity > 0,
         !is.na(UnitPrice), UnitPrice > 0) %>%
  # Customer-level analysis requires a customer identifier.
  filter(!is.na(CustomerID), CustomerID != "") %>%
  mutate(
    Revenue = Quantity * UnitPrice,
    InvoiceDate = as.POSIXct(InvoiceDate),
    InvoiceDate_Date = as.Date(InvoiceDate),
    Year = year(InvoiceDate),
    Month = month(InvoiceDate),
    Country = as.character(Country)
  )

cat("Rows after cleaning:", nrow(retail_clean), "\n")
cat("Columns after cleaning:", ncol(retail_clean), "\n")

# Save cleaned dataset
write_csv(retail_clean, file.path(output_dir, "cleaned_online_retail.csv"))

# -----------------------------
# 4. Before / after summary
# -----------------------------
missing_after <- data.frame(
  Variable = names(retail_clean),
  Missing_Count = sapply(retail_clean, function(x) sum(is.na(x))),
  Missing_Percentage = round(sapply(retail_clean, function(x) mean(is.na(x)) * 100), 4)
)

write_csv(missing_before, file.path(output_dir, "missing_summary_before.csv"))
write_csv(missing_after, file.path(output_dir, "missing_summary_after.csv"))

cleaning_summary <- data.frame(
  Metric = c(
    "Original rows",
    "Original columns",
    "Duplicate rows before cleaning",
    "Cancellation rows removed",
    "Rows after cleaning",
    "Rows removed overall"
  ),
  Value = c(
    nrow(retail_raw),
    ncol(retail_raw),
    duplicate_count_before,
    cancellation_count,
    nrow(retail_clean),
    nrow(retail_raw) - nrow(retail_clean)
  )
)

write_csv(cleaning_summary, file.path(output_dir, "cleaning_summary.csv"))
print(cleaning_summary)

# -----------------------------
# 5. Derive relational tables
# -----------------------------
cat("\n[4] DERIVING RELATIONAL TABLES FROM THE SAME UCI SOURCE\n")

# Transaction table: one row per cleaned transaction line.
transactions <- retail_clean %>%
  select(InvoiceNo, StockCode, CustomerID, InvoiceDate,
         Quantity, UnitPrice, Revenue, Country)

# Product dimension: one row per StockCode.
products <- retail_clean %>%
  group_by(StockCode) %>%
  summarise(
    Description = first(Description),
    UnitPrice = median(UnitPrice, na.rm = TRUE),
    .groups = "drop"
  )

# Customer dimension: one row per CustomerID.
customers <- retail_clean %>%
  group_by(CustomerID) %>%
  summarise(
    Country = first(Country),
    .groups = "drop"
  )

cat("Transactions:", nrow(transactions), "\n")
cat("Products:", nrow(products), "\n")
cat("Customers:", nrow(customers), "\n")

write_csv(transactions, file.path(output_dir, "transactions_derived.csv"))
write_csv(products, file.path(output_dir, "products_derived.csv"))
write_csv(customers, file.path(output_dir, "customers_derived.csv"))

# -----------------------------
# 6. Joins / integration
# -----------------------------
cat("\n[5] DATA INTEGRATION USING LEFT JOIN\n")

# Join product information to transactions using StockCode.
integrated <- transactions %>%
  left_join(products, by = "StockCode", suffix = c("_transaction", "_product")) %>%
  left_join(customers, by = "CustomerID", suffix = c("", "_customer"))

# Restore one clear product description field.
integrated <- integrated %>%
  rename(ProductDescription = Description)

cat("Integrated rows:", nrow(integrated), "\n")
cat("Integrated columns:", ncol(integrated), "\n")

unmatched_products <- sum(is.na(integrated$ProductDescription))
unmatched_customers <- sum(is.na(integrated$Country_customer))

join_summary <- data.frame(
  Check = c(
    "Integrated rows",
    "Integrated columns",
    "Unmatched product records",
    "Unmatched customer records"
  ),
  Value = c(
    nrow(integrated),
    ncol(integrated),
    unmatched_products,
    unmatched_customers
  )
)

print(join_summary)
write_csv(join_summary, file.path(output_dir, "join_summary.csv"))

# Why left_join?
join_explanation <- paste(
  "LEFT JOIN is used because the cleaned transaction table is the base table.",
  "Every valid transaction should remain in the integrated analytical dataset.",
  "Product and customer attributes are added where matching keys exist.",
  "This preserves transaction coverage while making unmatched keys visible."
)
writeLines(join_explanation, file.path(output_dir, "join_justification.txt"))

write_csv(integrated, file.path(output_dir, "integrated_retail_data.csv"))

# -----------------------------
# 7. Revenue analysis
# -----------------------------
cat("\n[6] REVENUE ANALYSIS\n")

total_revenue <- sum(retail_clean$Revenue, na.rm = TRUE)

top_products <- retail_clean %>%
  group_by(StockCode, Description) %>%
  summarise(
    Revenue = sum(Revenue, na.rm = TRUE),
    Quantity = sum(Quantity, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue)) %>%
  slice_head(n = 5)

top_countries <- retail_clean %>%
  group_by(Country) %>%
  summarise(
    Revenue = sum(Revenue, na.rm = TRUE),
    Quantity = sum(Quantity, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue)) %>%
  slice_head(n = 5)

top_customers <- retail_clean %>%
  group_by(CustomerID) %>%
  summarise(
    Revenue = sum(Revenue, na.rm = TRUE),
    Transactions = n_distinct(InvoiceNo),
    Quantity = sum(Quantity, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue)) %>%
  slice_head(n = 5)

cat("Total revenue:", round(total_revenue, 2), "\n\n")
cat("Top 5 products:\n")
print(top_products)
cat("\nTop 5 countries:\n")
print(top_countries)
cat("\nTop 5 customers:\n")
print(top_customers)

write_csv(top_products, file.path(output_dir, "top_5_products.csv"))
write_csv(top_countries, file.path(output_dir, "top_5_countries.csv"))
write_csv(top_customers, file.path(output_dir, "top_5_customers.csv"))

# -----------------------------
# 8. Customer segmentation with case_when()
# -----------------------------
cat("\n[7] CUSTOMER SEGMENTATION\n")

customer_summary <- retail_clean %>%
  group_by(CustomerID, Country) %>%
  summarise(
    TotalRevenue = sum(Revenue, na.rm = TRUE),
    TotalTransactions = n_distinct(InvoiceNo),
    TotalQuantity = sum(Quantity, na.rm = TRUE),
    .groups = "drop"
  )

# Use revenue quartiles as transparent, data-driven thresholds.
q <- quantile(customer_summary$TotalRevenue,
              probs = c(0.25, 0.50, 0.75),
              na.rm = TRUE)

customer_segments <- customer_summary %>%
  mutate(
    Segment = case_when(
      TotalRevenue <= q[1] ~ "Low",
      TotalRevenue <= q[2] ~ "Medium",
      TotalRevenue <= q[3] ~ "High",
      TRUE ~ "Premium"
    )
  )

segment_summary <- customer_segments %>%
  group_by(Segment) %>%
  summarise(
    Customers = n(),
    Revenue = sum(TotalRevenue),
    Average_Revenue = mean(TotalRevenue),
    .groups = "drop"
  ) %>%
  arrange(factor(Segment, levels = c("Low", "Medium", "High", "Premium")))

print(segment_summary)

write_csv(customer_segments, file.path(output_dir, "customer_segments.csv"))
write_csv(segment_summary, file.path(output_dir, "customer_segment_summary.csv"))

thresholds <- data.frame(
  Threshold = c("25th percentile", "50th percentile", "75th percentile"),
  Revenue = as.numeric(q)
)
write_csv(thresholds, file.path(output_dir, "segmentation_thresholds.csv"))

# -----------------------------
# 9. Market performance
# -----------------------------
market_summary <- retail_clean %>%
  group_by(Country) %>%
  summarise(
    Revenue = sum(Revenue),
    Transactions = n_distinct(InvoiceNo),
    Customers = n_distinct(CustomerID),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue))

high_market <- market_summary %>% slice_head(n = 1)
low_market <- market_summary %>% slice_tail(n = 1)

cat("\nHighest-performing market:\n")
print(high_market)

cat("\nLowest-revenue market among countries represented after cleaning:\n")
print(low_market)

write_csv(market_summary, file.path(output_dir, "market_performance.csv"))

# -----------------------------
# 10. Visualizations
# -----------------------------
cat("\n[8] CREATING VISUALIZATIONS\n")

p1 <- ggplot(top_countries, aes(x = reorder(Country, Revenue), y = Revenue)) +
  geom_col() +
  coord_flip() +
  labs(
    title = "Top 5 Countries by Revenue",
    x = "Country",
    y = "Revenue"
  ) +
  theme_minimal()

ggsave(
  file.path(output_dir, "top_5_countries_revenue.png"),
  p1, width = 9, height = 6, dpi = 150
)

p2 <- ggplot(segment_summary, aes(x = Segment, y = Revenue)) +
  geom_col() +
  labs(
    title = "Revenue by Customer Segment",
    x = "Customer Segment",
    y = "Revenue"
  ) +
  theme_minimal()

ggsave(
  file.path(output_dir, "customer_segment_revenue.png"),
  p2, width = 9, height = 6, dpi = 150
)

p3 <- ggplot(
  market_summary %>% slice_max(order_by = Revenue, n = 10),
  aes(x = reorder(Country, Revenue), y = Revenue)
) +
  geom_col() +
  coord_flip() +
  labs(
    title = "Top 10 Markets by Revenue",
    x = "Country",
    y = "Revenue"
  ) +
  theme_minimal()

ggsave(
  file.path(output_dir, "top_10_markets.png"),
  p3, width = 9, height = 7, dpi = 150
)

# -----------------------------
# 11. skimr summary
# -----------------------------
cat("\n[9] SKIMR SUMMARY\n")

skim_output <- capture.output(
  skim(retail_clean)
)

writeLines(skim_output, file.path(output_dir, "skimr_output.txt"))

# -----------------------------
# 12. SQLite database
# -----------------------------
cat("\n[10] SQLITE DATABASE\n")

db_path <- file.path(output_dir, "retail_sales.sqlite")

if (file.exists(db_path)) file.remove(db_path)

con <- dbConnect(RSQLite::SQLite(), db_path)

# Store the main cleaned analytical table.
dbWriteTable(con, "retail_sales", retail_clean, overwrite = TRUE)

# Store derived relational tables as well.
dbWriteTable(con, "transactions", transactions, overwrite = TRUE)
dbWriteTable(con, "products", products, overwrite = TRUE)
dbWriteTable(con, "customers", customers, overwrite = TRUE)
dbWriteTable(con, "customer_segments", customer_segments, overwrite = TRUE)

# Query 1: Top 5 customers
sql_top_customers <- "
SELECT
    CustomerID,
    ROUND(SUM(Revenue), 2) AS TotalRevenue
FROM retail_sales
GROUP BY CustomerID
ORDER BY TotalRevenue DESC
LIMIT 5;
"

# Query 2: Revenue by country
sql_country_revenue <- "
SELECT
    Country,
    ROUND(SUM(Revenue), 2) AS TotalRevenue
FROM retail_sales
GROUP BY Country
ORDER BY TotalRevenue DESC;
"

# Query 3: Revenue by product
sql_product_revenue <- "
SELECT
    StockCode,
    Description,
    ROUND(SUM(Revenue), 2) AS TotalRevenue,
    SUM(Quantity) AS TotalQuantity
FROM retail_sales
GROUP BY StockCode, Description
ORDER BY TotalRevenue DESC
LIMIT 10;
"

query_top_customers <- dbGetQuery(con, sql_top_customers)
query_country_revenue <- dbGetQuery(con, sql_country_revenue)
query_product_revenue <- dbGetQuery(con, sql_product_revenue)

cat("\nSQL — Top 5 customers:\n")
print(query_top_customers)

cat("\nSQL — Top countries:\n")
print(head(query_country_revenue, 10))

cat("\nSQL — Top products:\n")
print(query_product_revenue)

writeLines(sql_top_customers, file.path(output_dir, "sql_top_5_customers.sql"))
writeLines(sql_country_revenue, file.path(output_dir, "sql_revenue_by_country.sql"))
writeLines(sql_product_revenue, file.path(output_dir, "sql_top_products.sql"))

write_csv(query_top_customers, file.path(output_dir, "sql_result_top_5_customers.csv"))
write_csv(query_country_revenue, file.path(output_dir, "sql_result_revenue_by_country.csv"))
write_csv(query_product_revenue, file.path(output_dir, "sql_result_top_products.csv"))

# -----------------------------
# 13. Three business insights
# -----------------------------
cat("\n[11] BUSINESS INSIGHTS\n")

best_product <- top_products %>% slice(1)
best_customer <- top_customers %>% slice(1)
best_country <- top_countries %>% slice(1)
best_segment <- segment_summary %>% arrange(desc(Revenue)) %>% slice(1)

insight1 <- paste0(
  "1. MARKET: ", best_country$Country,
  " is the highest-revenue country in the cleaned dataset, generating approximately ",
  round(best_country$Revenue, 2),
  " in revenue. This market should receive priority for retention and growth initiatives."
)

insight2 <- paste0(
  "2. PRODUCT: StockCode ", best_product$StockCode,
  " (", best_product$Description,
  ") is the top product by revenue, contributing approximately ",
  round(best_product$Revenue, 2),
  ". This product can be prioritized in merchandising and cross-selling."
)

insight3 <- paste0(
  "3. CUSTOMER SEGMENT: The ", best_segment$Segment,
  " segment contributes the largest aggregate revenue at approximately ",
  round(best_segment$Revenue, 2),
  ". Marketing resources can be focused on retaining this segment and moving lower-value customers upward."
)

insights <- c(insight1, insight2, insight3)

cat("\n")
cat(paste(insights, collapse = "\n\n"))
cat("\n")

writeLines(insights, file.path(output_dir, "three_business_insights.txt"))

# -----------------------------
# 14. Final report
# -----------------------------
report_lines <- c(
  "# Assignment 3 — Module 2: Retail Analytics",
  "",
  "## Dataset",
  "UCI Online Retail dataset supplied by the student.",
  "",
  "## Cleaning",
  paste0("- Original rows: ", nrow(retail_raw)),
  paste0("- Rows after cleaning: ", nrow(retail_clean)),
  paste0("- Duplicate rows detected before cleaning: ", duplicate_count_before),
  paste0("- Cancellation rows identified: ", cancellation_count),
  "- Removed invalid/non-positive quantities and prices.",
  "- Removed records without required customer information for customer-level analysis.",
  "- Revenue calculated as Quantity × UnitPrice.",
  "",
  "## Integration",
  "Transaction, product and customer tables were derived from the same UCI source.",
  "LEFT JOIN was used so the transaction table remained the analytical base.",
  "",
  "## Revenue",
  paste0("- Total cleaned revenue: ", round(total_revenue, 2)),
  paste0("- Highest-revenue country: ", best_country$Country),
  paste0("- Highest-revenue product: ", best_product$Description),
  paste0("- Highest-revenue customer ID: ", best_customer$CustomerID),
  "",
  "## Customer Segmentation",
  "Customers were segmented using revenue quartiles and dplyr::case_when().",
  "",
  "## SQLite",
  "The retail_sales, transactions, products, customers and customer_segments tables were stored in retail_sales.sqlite.",
  "",
  "## Three Business Insights",
  insights
)

writeLines(report_lines, file.path(output_dir, "ASSIGNMENT_3_REPORT.md"))

# -----------------------------
# 15. Close database
# -----------------------------
dbDisconnect(con)

cat("\n========================================\n")
cat("ASSIGNMENT 3 COMPLETE\n")
cat("Outputs saved to:", output_dir, "\n")
cat("========================================\n")

print(list.files(output_dir))
'''
r_script_path = "/content/assignment3_complete.R"

with open(r_script_path, "w", encoding="utf-8") as f:
    f.write(r_code)

print("R script created:", r_script_path)
print("Running the complete R workflow...")

import subprocess, os

env = os.environ.copy()
env["UCI_EXCEL_PATH"] = excel_path

result = subprocess.run(
    ["Rscript", r_script_path],
    env=env,
    text=True,
    capture_output=True
)

print(result.stdout)

if result.returncode != 0:
    print("===== R ERROR =====")
    print(result.stderr)
    raise RuntimeError("The R workflow failed. See the R error above.")

print("✓ Complete R workflow finished successfully.")


R script created: /content/assignment3_complete.R
Running the complete R workflow...

ASSIGNMENT 3 — UCI ONLINE RETAIL
Input: /content/uci_online_retail/Online Retail.xlsx 

[1] IMPORTING DATA
Rows: 541909 
Columns: 8 

Column names:
[1] "InvoiceNo"   "StockCode"   "Description" "Quantity"    "InvoiceDate"
[6] "UnitPrice"   "CustomerID"  "Country"    

First 6 rows:
# A tibble: 6 × 8
  InvoiceNo StockCode Description         Quantity InvoiceDate         UnitPrice
  <chr>     <chr>     <chr>                  <dbl> <dttm>                  <dbl>
1 536365    85123A    WHITE HANGING HEAR…        6 2010-12-01 08:26:00      2.55
2 536365    71053     WHITE METAL LANTERN        6 2010-12-01 08:26:00      3.39
3 536365    84406B    CREAM CUPID HEARTS…        8 2010-12-01 08:26:00      2.75
4 536365    84029G    KNITTED UNION FLAG…        6 2010-12-01 08:26:00      3.39
5 536365    84029E    RED WOOLLY HOTTIE …        6 2010-12-01 08:26:00      3.39
6 536365    22752     SET 7 BABUSHKA NES…     

In [7]:
# CELL 5 — Inspect the generated outputs
import os, glob

output_dir = "/content/Assignment_3_Final_Deliverables"

print("Generated deliverables:\n")
for f in sorted(os.listdir(output_dir)):
    path = os.path.join(output_dir, f)
    size = os.path.getsize(path)
    print(f"✓ {f:45s} {size/1024:.1f} KB")


Generated deliverables:

✓ ASSIGNMENT_3_REPORT.md                        1.6 KB
✓ cleaned_online_retail.csv                     44028.6 KB
✓ cleaning_summary.csv                          0.2 KB
✓ customer_segment_revenue.png                  24.5 KB
✓ customer_segment_summary.csv                  0.2 KB
✓ customer_segments.csv                         173.7 KB
✓ customers_derived.csv                         86.2 KB
✓ integrated_retail_data.csv                    44327.7 KB
✓ join_justification.txt                        0.3 KB
✓ join_summary.csv                              0.1 KB
✓ market_performance.csv                        0.9 KB
✓ missing_summary_after.csv                     0.2 KB
✓ missing_summary_before.csv                    0.2 KB
✓ products_derived.csv                          140.2 KB
✓ retail_sales.sqlite                           64216.0 KB
✓ segmentation_thresholds.csv                   0.1 KB
✓ skimr_output.txt                              2.8 KB
✓ sql_result_revenue_b

In [8]:
# CELL 6 — Display the three business insights
insight_path = "/content/Assignment_3_Final_Deliverables/three_business_insights.txt"

with open(insight_path, "r", encoding="utf-8") as f:
    print(f.read())


1. MARKET: United Kingdom is the highest-revenue country in the cleaned dataset, generating approximately 7285024.64 in revenue. This market should receive priority for retention and growth initiatives.
2. PRODUCT: StockCode 23843 (PAPER CRAFT , LITTLE BIRDIE) is the top product by revenue, contributing approximately 168469.6. This product can be prioritized in merchandising and cross-selling.
3. CUSTOMER SEGMENT: The Premium segment contributes the largest aggregate revenue at approximately 7037623.49. Marketing resources can be focused on retaining this segment and moving lower-value customers upward.



In [9]:
# CELL 7 — Display the main SQL results
import pandas as pd, os

base = "/content/Assignment_3_Final_Deliverables"

print("TOP 5 CUSTOMERS")
display(pd.read_csv(os.path.join(base, "sql_result_top_5_customers.csv")))

print("TOP COUNTRIES")
display(pd.read_csv(os.path.join(base, "sql_result_revenue_by_country.csv")).head(10))

print("TOP PRODUCTS")
display(pd.read_csv(os.path.join(base, "sql_result_top_products.csv")))


TOP 5 CUSTOMERS


,CustomerID,TotalRevenue
0,14646,280206.02
1,18102,259657.30
2,17450,194390.79
3,16446,168472.50
4,14911,143711.17


TOP COUNTRIES


,Country,TotalRevenue
0,United Kingdom,7285024.64
1,Netherlands,285446.34
2,EIRE,265262.46
3,Germany,228678.40
4,France,208934.31
5,Australia,138453.81
6,Spain,61558.56
7,Switzerland,56443.95
8,Belgium,41196.34
9,Sweden,38367.83


TOP PRODUCTS


,StockCode,Description,TotalRevenue,TotalQuantity
0,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995
1,22423,REGENCY CAKESTAND 3 TIER,142264.75,12374
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,100392.10,36706
3,85099B,JUMBO BAG RED RETROSPOT,85040.54,46078
4,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.73,77916
5,POST,POSTAGE,77803.96,3120
6,47566,PARTY BUNTING,68785.23,15279
7,84879,ASSORTED COLOUR BIRD ORNAMENT,56413.03,35263
8,M,Manual,53419.93,6933
9,23084,RABBIT NIGHT LIGHT,51251.24,27153


In [10]:
# CELL 8 — Create ONE final submission ZIP
# The ZIP contains the complete R script, cleaned data, analysis outputs,
# SQL queries, SQLite database, plots, report and this notebook.

import os, zipfile, shutil

output_dir = "/content/Assignment_3_Final_Deliverables"
notebook_path = "/content/Assignment_3_Module_2_UCI_Online_Retail_COMPLETE.ipynb"

# Save a copy of the notebook into the output directory after this notebook
# has been written by Colab. If it is not available yet, the notebook download
# below still works from the generated file.

zip_path = "/content/Assignment_3_Module_2_FINAL_SUBMISSION.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files_ in os.walk(output_dir):
        for file in files_:
            full = os.path.join(root, file)
            arc = os.path.relpath(full, output_dir)
            z.write(full, arcname=arc)

print("✓ Final deliverables ZIP created:")
print(zip_path)
print("\nZIP contents:")
with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        print("✓", name)


✓ Final deliverables ZIP created:
/content/Assignment_3_Module_2_FINAL_SUBMISSION.zip

ZIP contents:
✓ sql_top_products.sql
✓ sql_top_5_customers.sql
✓ top_10_markets.png
✓ three_business_insights.txt
✓ retail_sales.sqlite
✓ transactions_derived.csv
✓ customer_segments.csv
✓ join_summary.csv
✓ customer_segment_summary.csv
✓ sql_revenue_by_country.sql
✓ join_justification.txt
✓ customers_derived.csv
✓ cleaned_online_retail.csv
✓ top_5_countries.csv
✓ customer_segment_revenue.png
✓ skimr_output.txt
✓ top_5_customers.csv
✓ top_5_products.csv
✓ missing_summary_before.csv
✓ sql_result_revenue_by_country.csv
✓ sql_result_top_products.csv
✓ market_performance.csv
✓ top_5_countries_revenue.png
✓ missing_summary_after.csv
✓ sql_result_top_5_customers.csv
✓ integrated_retail_data.csv
✓ products_derived.csv
✓ segmentation_thresholds.csv
✓ ASSIGNMENT_3_REPORT.md
✓ cleaning_summary.csv


In [11]:
# CELL 9 — Download the final deliverables ZIP
from google.colab import files

files.download("/content/Assignment_3_Module_2_FINAL_SUBMISSION.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>